# Airline Review Sentiment Analysis with MongoDB


In [ ]:
"""
Airline Review Sentiment Analysis with MongoDB

This script loads airline review data into MongoDB, cleans review text,
calculates sentiment scores, identifies complaint categories, and creates
visualizations for customer satisfaction analysis.

Requirements:
1. Replace YOUR_PASSWORD and YOUR_CONNECTION_STRING with your own MongoDB Atlas credentials.
2. Update CSV_FILE_PATH to point to your local dataset.
3. Create an images/ folder if you want to save chart outputs.
"""


## 0. Import libraries

In [ ]:
import re
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.probability import FreqDist
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from pymongo import MongoClient
from scipy.stats import pearsonr
from textblob import TextBlob


## 1. Connect to MongoDB

## Replace the placeholder values below with your own MongoDB Atlas credentials.

In [ ]:
# Replace the placeholder values below with your own MongoDB Atlas credentials.
pwd = "YOUR_PASSWORD"


def get_database():
    # Replace with your MongoDB Atlas connection string.
    CONNECTION_STRING = "YOUR_CONNECTION_STRING"

    # Create a connection to the MongoDB Atlas cluster.
    client = MongoClient(CONNECTION_STRING)

    return client["Airline_Performance"]


# Connect to the database.
dbname = get_database()

# Create collection references in the Airline_Performance database.
reviews_collection = dbname["Airline_Reviews"]
metadata_collection = dbname["Airline_Metadata"]
sentiment_collection = dbname["AirlineReviews_Sentiment"]
categories_collection = dbname["Sentiment_Categories"]


## 2. Load and store data

In [ ]:
CSV_FILE_PATH = "data/Airline_review.csv"
IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)


def load_airline_data(csv_file_path):
    """Load the airline review CSV and remove duplicate rows."""
    df = pd.read_csv(csv_file_path, encoding="utf-8")
    df_clean = df.drop_duplicates().reset_index(drop=True)
    return df_clean


def insert_reviews_into_mongodb(df_clean):
    """Split the dataset into review text and metadata collections."""
    review_records = df_clean[["Airline Name", "Review"]].to_dict("records")

    metadata_columns = [
        "Airline Name",
        "Overall_Rating",
        "Review_Title",
        "Review Date",
        "Verified",
        "Aircraft",
        "Type Of Traveller",
        "Seat Type",
        "Route",
        "Date Flown",
        "Seat Comfort",
        "Cabin Staff Service",
        "Food & Beverages",
        "Ground Service",
        "Inflight Entertainment",
        "Wifi & Connectivity",
        "Value For Money",
        "Recommended",
    ]
    metadata_records = df_clean[metadata_columns].to_dict("records")

    # Clear existing documents before inserting new records.
    reviews_collection.delete_many({})
    metadata_collection.delete_many({})

    reviews_collection.insert_many(review_records)
    metadata_collection.insert_many(metadata_records)

    print(f"Inserted {len(review_records):,} review records into MongoDB.")
    print(f"Inserted {len(metadata_records):,} metadata records into MongoDB.")


## 3. Explore MongoDB records

In [ ]:
def summarize_airline_review_counts():
    """Count reviews by airline and print the top airlines by review volume."""
    start = time.time()

    pipeline = [
        {"$group": {"_id": "$Airline Name", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}},
    ]

    airline_counts = list(reviews_collection.aggregate(pipeline))
    unique_airline_count = reviews_collection.distinct("Airline Name")

    print(f"Number of unique airlines: {len(unique_airline_count):,}")
    print("\nTop 10 airlines by number of reviews:")
    for doc in airline_counts[:10]:
        print(f"{doc['_id']}: {doc['count']:,} reviews")

    end = time.time()
    print(f"Review count query time: {end - start:.4f} seconds")

    return airline_counts


## 4. Clean and preprocess review text

In [ ]:
IMPORTANT_STOPWORDS = {
    "not",
    "no",
    "never",
    "nor",
    "can",
    "will",
    "should",
    "would",
    "could",
    "might",
    "must",
    "only",
    "very",
}


def clean_review_text(review, stop_words):
    """Clean one review by lowercasing, removing punctuation, and removing stopwords."""
    if not isinstance(review, str):
        return []

    review = review.lower()

    # Expand contractions ending in n't before removing punctuation.
    review = re.sub(r"(\w+)n\'t", r"\1 not", review)

    # Keep alphabetic words only.
    words = review.split()
    alphabetic_words = [word for word in words if re.fullmatch(r"^[a-zA-Z]+$", word)]

    # Remove stopwords while keeping words that can affect sentiment.
    cleaned_words = [word for word in alphabetic_words if word not in stop_words]

    return cleaned_words


def get_wordnet_pos(tag):
    """Map NLTK part-of-speech tags to WordNet tags for lemmatization."""
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN


def lemmatize_tokens(token_list):
    """Reduce tokens to their base forms using part-of-speech tags."""
    lemmatizer = WordNetLemmatizer()
    tagged_tokens = pos_tag(token_list)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged_tokens]


def preprocess_reviews():
    """Load reviews from MongoDB, clean the text, and lemmatize tokens."""
    start = time.time()

    records = list(reviews_collection.find({}, {"_id": 0, "Review": 1}))
    reviews_df = pd.DataFrame(records)

    stop_words = set(stopwords.words("english"))
    stop_words = stop_words.difference(IMPORTANT_STOPWORDS)

    reviews_df["Cleaned_Review"] = reviews_df["Review"].apply(
        lambda review: clean_review_text(review, stop_words)
    )
    reviews_df["Lemmatized_Review"] = reviews_df["Cleaned_Review"].apply(lemmatize_tokens)

    end = time.time()
    print(f"Preprocessing and lemmatization time: {end - start:.4f} seconds")
    print(reviews_df[["Review", "Lemmatized_Review"]].head())

    return reviews_df


## 5. Calculate sentiment scores

In [ ]:
def calculate_sentiment_scores(preprocessed_reviews_df):
    """Calculate TextBlob sentiment scores and store them in MongoDB."""
    start = time.time()

    airline_records = list(reviews_collection.find({}, {"_id": 0, "Airline Name": 1}))
    airline_df = pd.DataFrame(airline_records)

    sentiment_df = pd.concat(
        [airline_df, preprocessed_reviews_df[["Lemmatized_Review"]]], axis=1
    )

    sentiment_df["Cleaned_Review_String"] = sentiment_df["Lemmatized_Review"].apply(
        lambda tokens: " ".join(tokens)
    )

    sentiment_df["sentiment_score"] = sentiment_df["Cleaned_Review_String"].apply(
        lambda review: TextBlob(review).sentiment.polarity
    )

    # Convert TextBlob polarity score from [-1, 1] to a 0-10 scale.
    sentiment_df["scaled_sentiment_score"] = sentiment_df["sentiment_score"].apply(
        lambda score: round(((score + 1) / 2) * 10, 2)
    )

    final_sentiment_df = sentiment_df[
        ["Airline Name", "Lemmatized_Review", "scaled_sentiment_score"]
    ].rename(columns={"Lemmatized_Review": "Cleaned_Review"})

    sentiment_collection.delete_many({})
    sentiment_collection.insert_many(final_sentiment_df.to_dict("records"))

    end = time.time()
    print(f"Sentiment analysis time: {end - start:.4f} seconds")
    print(final_sentiment_df.head())

    return final_sentiment_df


## 6. Compare sentiment scores with original ratings

In [ ]:
def calculate_rating_sentiment_correlation():
    """Compare calculated sentiment scores with original overall ratings."""
    sentiment_records = list(
        sentiment_collection.find(
            {}, {"_id": 0, "Airline Name": 1, "scaled_sentiment_score": 1}
        )
    )
    metadata_records = list(
        metadata_collection.find({}, {"_id": 0, "Airline Name": 1, "Overall_Rating": 1})
    )

    sentiment_df = pd.DataFrame(sentiment_records)
    metadata_df = pd.DataFrame(metadata_records)

    merged_df = pd.merge(sentiment_df, metadata_df, on="Airline Name", how="inner")
    merged_df["scaled_sentiment_score"] = pd.to_numeric(
        merged_df["scaled_sentiment_score"], errors="coerce"
    )
    merged_df["Overall_Rating"] = pd.to_numeric(
        merged_df["Overall_Rating"], errors="coerce"
    )
    merged_df = merged_df.dropna(subset=["scaled_sentiment_score", "Overall_Rating"])

    pearson_r, _ = pearsonr(
        merged_df["scaled_sentiment_score"], merged_df["Overall_Rating"]
    )

    print(f"Pearson correlation between sentiment score and rating: {pearson_r:.2f}")
    return pearson_r


## 7. Identify complaint categories in negative reviews

In [ ]:
COMPLAINT_CATEGORIES = {
    "Delays & Cancellations": [
        "delayed",
        "delay",
        "late",
        "waiting",
        "hours",
        "scheduled",
        "arrival",
        "departure",
        "cancelled",
        "cancel",
    ],
    "Baggage": ["luggage", "bags", "baggage", "lost", "carry", "bag"],
    "Customer Service": [
        "staff",
        "crew",
        "attendants",
        "rude",
        "help",
        "customer",
        "agent",
        "care",
        "service",
    ],
    "Boarding": ["boarding", "check", "gate", "security", "counter", "line", "desk"],
    "Seating Comfort": ["seat", "seats", "cabin", "class", "leg", "row"],
    "Food & Beverages": ["food", "meal", "meals", "drinks", "drink", "water"],
    "Refunds": ["refund", "paid", "pay", "price", "cost", "money"],
    "Flight Experience": [
        "plane",
        "flight",
        "flights",
        "aircraft",
        "comfort",
        "entertainment",
        "inflight",
        "clean",
    ],
    "Airline Communication": ["told", "said", "information", "informed", "contact"],
    "Rebooking": ["booked", "booking", "changed", "change", "next"],
    "Overall Issues": [
        "experience",
        "company",
        "worst",
        "problem",
        "issue",
        "complaint",
        "avoid",
        "poor",
    ],
}


def review_category_flags(cleaned_review):
    """Assign complaint categories based on whether category keywords appear in a review."""
    review_categories = set()

    for category, keywords in COMPLAINT_CATEGORIES.items():
        if any(word in keywords for word in cleaned_review):
            review_categories.add(category)

    return sorted(review_categories)


def categorize_negative_reviews(sentiment_df):
    """Filter negative reviews and assign complaint categories."""
    start = time.time()

    negative_reviews_df = sentiment_df[sentiment_df["scaled_sentiment_score"] < 5.0].copy()
    negative_reviews_df["complaint_categories"] = negative_reviews_df[
        "Cleaned_Review"
    ].apply(review_category_flags)

    categories_collection.delete_many({})
    categories_collection.insert_many(negative_reviews_df.to_dict("records"))

    end = time.time()
    print(f"Complaint categorization time: {end - start:.4f} seconds")
    print(negative_reviews_df.head())

    return negative_reviews_df


## 8. Explore frequent words in negative reviews

In [ ]:
def show_frequent_words(sentiment_df, top_n=30):
    """Display the most frequent words across cleaned reviews."""
    cleaned_text = " ".join(
        sentiment_df["Cleaned_Review"].apply(lambda tokens: " ".join(tokens))
    )
    tokens = word_tokenize(cleaned_text)
    frequency_distribution = FreqDist(tokens)

    print(f"Top {top_n} most common words:")
    print(frequency_distribution.most_common(top_n))

    return frequency_distribution


## 9. Create visualizations

In [ ]:
def plot_complaint_category_distribution(negative_reviews_df):
    """Create a bar chart showing how often each complaint category appears."""
    start = time.time()

    all_categories = []
    for category_list in negative_reviews_df["complaint_categories"]:
        all_categories.extend(category_list)

    category_counts = Counter(all_categories)
    category_df = pd.DataFrame.from_dict(
        category_counts, orient="index", columns=["count"]
    )
    category_df["percent"] = (category_df["count"] / len(negative_reviews_df)) * 100
    category_df = category_df.sort_values("percent", ascending=False)

    plt.figure(figsize=(10, 6))
    plt.bar(category_df.index, category_df["percent"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("% of Negative Reviews")
    plt.title("Distribution of Complaint Categories")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "complaint_category_distribution.png", dpi=300)
    plt.show()

    end = time.time()
    print(f"Simple bar chart creation time: {end - start:.4f} seconds")

    return category_df


def plot_airline_complaint_breakdown(negative_reviews_df):
    """Create a stacked bar chart of complaint categories for the top 10 airlines."""
    start = time.time()

    top_airlines = negative_reviews_df["Airline Name"].value_counts().head(10).index.tolist()

    records = []
    for airline in top_airlines:
        airline_subset = negative_reviews_df[negative_reviews_df["Airline Name"] == airline]
        for category_list in airline_subset["complaint_categories"]:
            for category in category_list:
                records.append(
                    {"Airline Name": airline, "Complaint Category": category}
                )

    flat_df = pd.DataFrame(records)
    grouped_df = (
        flat_df.groupby(["Airline Name", "Complaint Category"])
        .size()
        .reset_index(name="Count")
    )
    grouped_df["Percentage"] = grouped_df.groupby("Airline Name")["Count"].transform(
        lambda x: x / x.sum() * 100
    )

    fig = px.bar(
        grouped_df,
        x="Airline Name",
        y="Percentage",
        color="Complaint Category",
        title="Complaint Categories for the 10 Airlines with the Most Negative Reviews",
        labels={"Percentage": "% of Complaints"},
        height=600,
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.write_html(IMAGES_DIR / "airline_complaint_breakdown.html")
    fig.show()

    end = time.time()
    print(f"Stacked bar chart creation time: {end - start:.4f} seconds")

    return grouped_df


def classify_opinion(score):
    """Convert scaled sentiment score into a satisfaction label."""
    if score >= 6.67:
        return "satisfied"
    if score >= 3.33:
        return "neutral"
    return "dissatisfied"


def plot_satisfaction_by_seat_type():
    """Create a grouped bar chart showing satisfaction by seat type."""
    start = time.time()

    sentiment_records = list(
        sentiment_collection.find(
            {}, {"_id": 0, "Airline Name": 1, "scaled_sentiment_score": 1}
        )
    )
    metadata_records = list(
        metadata_collection.find({}, {"_id": 0, "Airline Name": 1, "Seat Type": 1})
    )

    sentiment_df = pd.DataFrame(sentiment_records)
    metadata_df = pd.DataFrame(metadata_records)

    merged_df = pd.merge(sentiment_df, metadata_df, on="Airline Name", how="inner")
    merged_df["scaled_sentiment_score"] = pd.to_numeric(
        merged_df["scaled_sentiment_score"], errors="coerce"
    )
    merged_df = merged_df.dropna(subset=["scaled_sentiment_score", "Seat Type"])
    merged_df = merged_df[merged_df["Seat Type"] != ""]

    merged_df["sentiment"] = merged_df["scaled_sentiment_score"].apply(classify_opinion)

    counts_df = (
        merged_df.groupby(["Seat Type", "sentiment"]).size().reset_index(name="count")
    )
    counts_df["total_seat"] = counts_df.groupby("Seat Type")["count"].transform("sum")
    counts_df["percentage"] = counts_df["count"] / counts_df["total_seat"] * 100

    pivot_df = counts_df.pivot(
        index="Seat Type", columns="sentiment", values="percentage"
    ).fillna(0)
    pivot_df = pivot_df[[col for col in ["satisfied", "neutral", "dissatisfied"] if col in pivot_df.columns]]

    ax = pivot_df.plot(kind="bar", figsize=(12, 7))
    ax.set_title("Normalized Customer Satisfaction by Seat Type")
    ax.set_ylabel("Percentage of Reviews (%)")
    ax.set_xlabel("Seat Type")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "satisfaction_by_seat_type.png", dpi=300)
    plt.show()

    end = time.time()
    print(f"Grouped bar chart creation time: {end - start:.4f} seconds")

    return counts_df


def clean_review_date(date_value):
    """Convert review date strings like '1st January 2020' into datetime values."""
    if not isinstance(date_value, str):
        return None

    date_clean = re.sub(r"(\d{1,2})(st|nd|rd|th)", r"\1", date_value)

    try:
        return datetime.strptime(date_clean, "%d %B %Y")
    except ValueError:
        return None


def plot_yearly_sentiment_trend():
    """Create a line chart showing yearly average customer sentiment over time."""
    sentiment_records = list(
        sentiment_collection.find(
            {}, {"_id": 0, "Airline Name": 1, "scaled_sentiment_score": 1}
        )
    )
    metadata_records = list(
        metadata_collection.find({}, {"_id": 0, "Airline Name": 1, "Review Date": 1})
    )

    sentiment_df = pd.DataFrame(sentiment_records)
    metadata_df = pd.DataFrame(metadata_records)

    time_df = pd.merge(sentiment_df, metadata_df, on="Airline Name", how="inner")
    time_df["scaled_sentiment_score"] = pd.to_numeric(
        time_df["scaled_sentiment_score"], errors="coerce"
    )
    time_df["Date"] = time_df["Review Date"].apply(clean_review_date)
    time_df = time_df.dropna(subset=["Date", "scaled_sentiment_score"])

    time_df["Year"] = pd.to_datetime(time_df["Date"]).dt.to_period("Y")
    yearly_avg = time_df.groupby("Year")["scaled_sentiment_score"].mean().reset_index()
    yearly_avg["Year"] = yearly_avg["Year"].dt.to_timestamp()

    plt.figure(figsize=(14, 6))
    plt.plot(yearly_avg["Year"], yearly_avg["scaled_sentiment_score"], marker="o")
    plt.title("Yearly Average Customer Satisfaction Over Time")
    plt.xlabel("Date")
    plt.ylabel("Average Sentiment Score")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "yearly_sentiment_trend.png", dpi=300)
    plt.show()

    return yearly_avg


## 10. Test MongoDB query performance

In [ ]:
def test_query_performance(airline_names):
    """Measure how long it takes to retrieve reviews for a list of airlines."""
    start = time.time()

    for airline in airline_names:
        list(
            reviews_collection.find(
                {"Airline Name": airline},
                {"_id": 0, "Review": 1},
            )
        )

    end = time.time()
    print(f"Query time for {len(airline_names)} airlines: {end - start:.4f} seconds")


## 11. Run full workflow

In [ ]:
if __name__ == "__main__":
    # Load CSV data and insert it into MongoDB.
    airline_df = load_airline_data(CSV_FILE_PATH)
    insert_reviews_into_mongodb(airline_df)

    # Explore records and review volume.
    summarize_airline_review_counts()

    # Clean reviews and calculate sentiment.
    preprocessed_df = preprocess_reviews()
    sentiment_df = calculate_sentiment_scores(preprocessed_df)
    calculate_rating_sentiment_correlation()

    # Identify complaint themes.
    show_frequent_words(sentiment_df, top_n=30)
    negative_reviews_df = categorize_negative_reviews(sentiment_df)

    # Create visualizations.
    plot_complaint_category_distribution(negative_reviews_df)
    plot_airline_complaint_breakdown(negative_reviews_df)
    plot_satisfaction_by_seat_type()
    plot_yearly_sentiment_trend()

    # Test MongoDB query performance.
    top_ten_airlines = [
        "Bangkok Airways",
        "Norwegian",
        "airBaltic",
        "Aeromexico",
        "Virgin America",
        "Asiana Airlines",
        "Volaris",
        "Lufthansa",
        "Garuda Indonesia",
        "Aeroflot Russian Airlines",
    ]
    test_query_performance(top_ten_airlines)
